# Alternative Detector Benchmark Launcher

This notebook is separate from `RunRFDETR_Solo_SingleClass_STATIC_Ucloud.ipynb`.

Goal: prepare a controlled validation-only benchmark for three RF-DETR alternatives:

- `DEIMv2-X`
- `EdgeCrafter ECDet-X`
- `D-FINE-X` with Objects365 pretraining

Defaults are intentionally safe: this notebook prints the plan only. Set `EXECUTE_SETUP=True` to clone/install/write configs, and set `RUN_TRAINING=True` only when ready to launch training.

In [1]:
from __future__ import annotations

import json
import os
import pathlib
import shlex
import subprocess
import sys
import textwrap
from dataclasses import dataclass

# -----------------------------------------------------------------------------
# Safety toggles
# -----------------------------------------------------------------------------
EXECUTE_SETUP = True   # clone repos, install deps, download public weights, write configs
RUN_TRAINING = True    # launch torchrun training commands
INSTALL_DEPS = True    # pip install -r requirements.txt inside each repo
DOWNLOAD_PUBLIC_WEIGHTS = True
WRITE_CONFIGS = True

# -----------------------------------------------------------------------------
# UCloud/project defaults
# -----------------------------------------------------------------------------
RUNTIME_PROFILE = os.environ.get("ALTDET_RUNTIME_PROFILE", "ucloud").strip().lower()
PROJECT_DIR = pathlib.Path(os.environ.get("PROJECT_DIR", "/work/projects/myproj/SOLO_Supervised_RFDETR/")).resolve()
DATASET_TWO_CLASS = pathlib.Path(os.environ.get(
    "DATASET_TWO_CLASS",
    str(PROJECT_DIR / "Stat_Dataset" / "QA-2025v1_TwoClass_OVR_V2_20260618-101346"),
)).resolve()

def detect_ucloud_user_base() -> str:
    work = pathlib.Path("/work")
    if not work.exists():
        return ""
    member = sorted(work.glob("Member Files:*"))
    if member:
        return member[0].name
    hashed = sorted([p for p in work.glob("*#*") if p.is_dir()])
    return hashed[0].name if hashed else ""

UCLOUD_USER_BASE = detect_ucloud_user_base()
WORK_ROOT = pathlib.Path(os.environ.get("ALTDET_WORK_ROOT", f"/work/{UCLOUD_USER_BASE}" if UCLOUD_USER_BASE else str(PROJECT_DIR.parent))).resolve()
OUTPUT_ROOT = pathlib.Path(os.environ.get("ALTDET_OUTPUT_ROOT", str(WORK_ROOT / "ALT_DETECTOR_BENCHMARK"))).resolve()
THIRD_PARTY_ROOT = pathlib.Path(os.environ.get("ALTDET_THIRD_PARTY_ROOT", str(WORK_ROOT / "third_party_detectors"))).resolve()
MODEL_ZOO_DIR = pathlib.Path(os.environ.get("ALTDET_MODEL_ZOO_DIR", str(WORK_ROOT / "Detector_Checkpoints"))).resolve()

CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3,4,5,6,7")
NPROC_PER_NODE = int(os.environ.get("ALTDET_NPROC_PER_NODE", "8"))
NUM_WORKERS = int(os.environ.get("ALTDET_NUM_WORKERS", "8"))
SEED = int(os.environ.get("ALTDET_SEED", "42"))
EPOCHS = int(os.environ.get("ALTDET_EPOCHS", "120"))
NUM_CLASSES = 2
CLASS_NAMES = ["Leucocyte", "Squamous Epithelial Cell"]

# Start at 640 because all three repos publish strong 640 configs. Later, test 896/1024 only if the first pass is stable.
INPUT_SIZE = int(os.environ.get("ALTDET_INPUT_SIZE", "640"))

# Use the detector training environment explicitly. The UCloud notebook kernel may be Python 3.13,
# which is too new for several detection repos and PyTorch/CUDA dependency pins.
DEFAULT_PROJECT_PYTHON = pathlib.Path("/work/CondaEnv/miniconda3/envs/aipowmic/bin/python")
DEFAULT_PYTHON_BIN = DEFAULT_PROJECT_PYTHON if DEFAULT_PROJECT_PYTHON.exists() else pathlib.Path(sys.executable)
PYTHON_BIN = pathlib.Path(os.environ.get("ALTDET_PYTHON", str(DEFAULT_PYTHON_BIN))).resolve()

print("PROJECT_DIR =", PROJECT_DIR)
print("DATASET_TWO_CLASS =", DATASET_TWO_CLASS)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("THIRD_PARTY_ROOT =", THIRD_PARTY_ROOT)
print("MODEL_ZOO_DIR =", MODEL_ZOO_DIR)
print("CUDA_VISIBLE_DEVICES =", CUDA_VISIBLE_DEVICES)
print("NPROC_PER_NODE =", NPROC_PER_NODE)
print("PYTHON_BIN =", PYTHON_BIN)
print("EXECUTE_SETUP =", EXECUTE_SETUP)
print("RUN_TRAINING =", RUN_TRAINING)


PROJECT_DIR = C:\work\projects\myproj\SOLO_Supervised_RFDETR
DATASET_TWO_CLASS = C:\work\projects\myproj\SOLO_Supervised_RFDETR\Stat_Dataset\QA-2025v1_TwoClass_OVR_V2_20260618-101346
OUTPUT_ROOT = C:\work\projects\myproj\ALT_DETECTOR_BENCHMARK
THIRD_PARTY_ROOT = C:\work\projects\myproj\third_party_detectors
MODEL_ZOO_DIR = C:\work\projects\myproj\Detector_Checkpoints
CUDA_VISIBLE_DEVICES = 0,1,2,3,4,5,6,7
NPROC_PER_NODE = 8
EXECUTE_SETUP = True
RUN_TRAINING = False


In [2]:
@dataclass
class Candidate:
    name: str
    family: str
    repo_url: str
    repo_dir: pathlib.Path
    work_subdir: str
    config_template: str
    config_out: str
    pretrained_name: str
    pretrained_url: str | None
    finetune_arg: str = "-t"
    notes: str = ""

CANDIDATES = [
    Candidate(
        name="deimv2_x",
        family="DEIMv2",
        repo_url="https://github.com/Intellindust-AI-Lab/DEIMv2.git",
        repo_dir=THIRD_PARTY_ROOT / "DEIMv2",
        work_subdir=".",
        config_template="configs/deimv2/deimv2_dinov3_x_coco.yml",
        config_out="configs/solo/deimv2_x_solo_twoclass.yml",
        pretrained_name="deimv2_dinov3_x_coco.pth",
        pretrained_url=None,
        notes="Needs DINOv3 backbone files in DEIMv2/ckpts. Download per DEIMv2 README/license requirements.",
    ),
    Candidate(
        name="edgecrafter_ecdet_x",
        family="EdgeCrafter",
        repo_url="https://github.com/Intellindust-AI-Lab/EdgeCrafter.git",
        repo_dir=THIRD_PARTY_ROOT / "EdgeCrafter",
        work_subdir="ecdetseg",
        config_template="configs/ecdet/ecdet_x.yml",
        config_out="configs/solo/ecdet_x_solo_twoclass.yml",
        pretrained_name="ecdet_x.pth",
        pretrained_url="https://github.com/capsule2077/edgecrafter/releases/download/edgecrafterv1/ecdet_x.pth",
        notes="EdgeCrafter docs use Python 3.11. Verify train.py args after cloning; it is DEIM/D-FINE-derived.",
    ),
    Candidate(
        name="dfine_x_o365",
        family="D-FINE",
        repo_url="https://github.com/Peterande/D-FINE.git",
        repo_dir=THIRD_PARTY_ROOT / "D-FINE",
        work_subdir=".",
        config_template="configs/dfine/custom/objects365/dfine_hgnetv2_x_obj2custom.yml",
        config_out="configs/solo/dfine_x_o365_solo_twoclass.yml",
        pretrained_name="dfine_x_obj365.pth",
        pretrained_url="https://github.com/Peterande/storage/releases/download/dfinev1.0/dfine_x_obj365.pth",
        notes="Uses Objects365 pretrained D-FINE-X checkpoint, then fine-tunes on our two-class COCO dataset.",
    ),
]

def python_bin_for(c: Candidate | None = None) -> pathlib.Path:
    if c is None:
        return PYTHON_BIN
    env_name = f"ALTDET_{c.name.upper()}_PYTHON"
    return pathlib.Path(os.environ.get(env_name, str(PYTHON_BIN))).resolve()

def python_version_for(c: Candidate | None = None) -> str:
    py = python_bin_for(c)
    probe = "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}'); print(sys.executable)"
    result = subprocess.run([str(py), "-c", probe], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise RuntimeError(f"Could not run selected Python {py}:\n{result.stdout}")
    return result.stdout.strip()

def assert_supported_python(c: Candidate | None = None) -> None:
    version_report = python_version_for(c)
    minor = version_report.splitlines()[0]
    if minor not in {"3.10", "3.11"}:
        name = c.name if c else "global"
        env_hint = f"ALTDET_{c.name.upper()}_PYTHON" if c else "ALTDET_PYTHON"
        raise RuntimeError(
            f"{name} is configured to use Python {minor}. These detector repos should be run from Python 3.10/3.11, "
            f"not the notebook/kernel Python. Create/select a detector env and set {env_hint}=/path/to/env/bin/python.\n"
            f"Selected interpreter report:\n{version_report}"
        )

def run(cmd: list[str] | str, cwd: pathlib.Path | None = None, execute: bool = False, capture: bool = False) -> None:
    if isinstance(cmd, list):
        printable = " ".join(shlex.quote(str(x)) for x in cmd)
    else:
        printable = cmd
    print("\n$", printable)
    if cwd:
        print("  cwd:", cwd)
    if execute:
        if capture:
            result = subprocess.run(
                cmd,
                cwd=str(cwd) if cwd else None,
                check=False,
                shell=isinstance(cmd, str),
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
            )
            if result.stdout:
                print(result.stdout)
            if result.returncode != 0:
                raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
        else:
            subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True, shell=isinstance(cmd, str))

def validate_coco_dataset(dataset_dir: pathlib.Path) -> None:
    print("\n[DATASET CHECK]", dataset_dir)
    for split in ["train", "valid", "test"]:
        ann = dataset_dir / split / "_annotations.coco.json"
        if not ann.exists():
            raise FileNotFoundError(f"Missing {ann}")
        data = json.loads(ann.read_text(encoding="utf-8"))
        print(split, "images=", len(data.get("images", [])), "annotations=", len(data.get("annotations", [])), "categories=", data.get("categories", []))

def candidate_work_dir(c: Candidate) -> pathlib.Path:
    return (c.repo_dir / c.work_subdir).resolve()

def weight_path(c: Candidate) -> pathlib.Path:
    env_name = f"ALTDET_{c.name.upper()}_WEIGHTS"
    return pathlib.Path(os.environ.get(env_name, str(MODEL_ZOO_DIR / c.pretrained_name))).resolve()

def dataset_yaml_text() -> str:
    return textwrap.dedent(f"""
    task: detection

    evaluator:
      type: CocoEvaluator
      iou_types: ['bbox']

    num_classes: {NUM_CLASSES}
    remap_mscoco_category: False

    train_dataloader:
      type: DataLoader
      dataset:
        type: CocoDetection
        img_folder: {DATASET_TWO_CLASS / 'train'}
        ann_file: {DATASET_TWO_CLASS / 'train' / '_annotations.coco.json'}
        return_masks: False
        transforms:
          type: Compose
          ops: ~
      shuffle: True
      num_workers: {NUM_WORKERS}
      drop_last: True
      collate_fn:
        type: BatchImageCollateFunction

    val_dataloader:
      type: DataLoader
      dataset:
        type: CocoDetection
        img_folder: {DATASET_TWO_CLASS / 'valid'}
        ann_file: {DATASET_TWO_CLASS / 'valid' / '_annotations.coco.json'}
        return_masks: False
        transforms:
          type: Compose
          ops: ~
      shuffle: False
      num_workers: {NUM_WORKERS}
      drop_last: False
      collate_fn:
        type: BatchImageCollateFunction
    """).strip() + "\n"

def write_candidate_config(c: Candidate) -> pathlib.Path:
    work_dir = candidate_work_dir(c)
    dataset_cfg = work_dir / "configs" / "dataset" / "solo_twoclass_detection.yml"
    dataset_cfg.parent.mkdir(parents=True, exist_ok=True)
    dataset_cfg.write_text(dataset_yaml_text(), encoding="utf-8")

    src = work_dir / c.config_template
    dst = work_dir / c.config_out
    if not src.exists():
        raise FileNotFoundError(f"Config template not found for {c.name}: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    text = src.read_text(encoding="utf-8")

    # The DEIM/RT-DETR-style repos use includes for dataset configs. This replacement is deliberately broad but local to a fresh third-party clone.
    text = text.replace("../dataset/coco_detection.yml", "../dataset/solo_twoclass_detection.yml")
    text = text.replace("../dataset/custom_detection.yml", "../dataset/solo_twoclass_detection.yml")
    text = text.replace("configs/dataset/coco_detection.yml", "configs/dataset/solo_twoclass_detection.yml")
    text = text.replace("configs/dataset/custom_detection.yml", "configs/dataset/solo_twoclass_detection.yml")

    extra = f"""

    # -------------------------------------------------------------------------
    # SOLO two-class benchmark overrides generated by this launcher.
    # Selection must use validation only. The clinically validated test split is
    # intentionally not configured here.
    # -------------------------------------------------------------------------
    output_dir: {OUTPUT_ROOT / c.name}
    eval_spatial_size: [{INPUT_SIZE}, {INPUT_SIZE}]
    epoches: {EPOCHS}
    num_classes: {NUM_CLASSES}
    remap_mscoco_category: False
    """
    dst.write_text(text.rstrip() + "\n" + textwrap.dedent(extra).lstrip(), encoding="utf-8")
    return dst

def train_command(c: Candidate) -> str:
    work_dir = candidate_work_dir(c)
    ckpt = weight_path(c)
    cfg = c.config_out
    cmd = [
        str(python_bin_for(c)),
        "-m", "torch.distributed.run",
        "--master_port", str(29500 + CANDIDATES.index(c)),
        "--nproc_per_node", str(NPROC_PER_NODE),
        "train.py",
        "-c", cfg,
        "--use-amp",
        "--seed", str(SEED),
    ]
    if ckpt.exists() or c.pretrained_url:
        cmd.extend([c.finetune_arg, str(ckpt)])
    return f"CUDA_VISIBLE_DEVICES={CUDA_VISIBLE_DEVICES} " + " ".join(shlex.quote(str(x)) for x in cmd)


In [3]:
# This cell prints and optionally performs setup. It does not train unless RUN_TRAINING=True.
validate_coco_dataset(DATASET_TWO_CLASS)

print("\n[CANDIDATES]")
for c in CANDIDATES:
    print(f"- {c.name}: {c.family}")
    print("  repo:", c.repo_url)
    print("  repo_dir:", c.repo_dir)
    print("  work_dir:", candidate_work_dir(c))
    print("  config_template:", c.config_template)
    print("  config_out:", c.config_out)
    print("  pretrained:", weight_path(c))
    print("  python:", python_bin_for(c))
    print("  notes:", c.notes)

if EXECUTE_SETUP:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    THIRD_PARTY_ROOT.mkdir(parents=True, exist_ok=True)
    MODEL_ZOO_DIR.mkdir(parents=True, exist_ok=True)

    for c in CANDIDATES:
        if not c.repo_dir.exists():
            run(["git", "clone", c.repo_url, str(c.repo_dir)], execute=True)
        else:
            run(["git", "pull", "--ff-only"], cwd=c.repo_dir, execute=True)

        if INSTALL_DEPS:
            assert_supported_python(c)
            req = c.repo_dir / "requirements.txt"
            if req.exists():
                run([str(python_bin_for(c)), "-m", "pip", "install", "-r", str(req)], cwd=c.repo_dir, execute=True, capture=True)

        if DOWNLOAD_PUBLIC_WEIGHTS and c.pretrained_url and not weight_path(c).exists():
            run([str(python_bin_for(c)), "-c", f"import urllib.request; urllib.request.urlretrieve({c.pretrained_url!r}, {str(weight_path(c))!r})"], execute=True)

        if WRITE_CONFIGS:
            cfg = write_candidate_config(c)
            print("[CONFIG] wrote", cfg)
else:
    print("\n[SETUP] EXECUTE_SETUP=False; no repos/configs/weights changed.")

print("\n[TRAINING COMMANDS - validation only]")
for c in CANDIDATES:
    print("\n#", c.name)
    print("cd", shlex.quote(str(candidate_work_dir(c))))
    print(train_command(c))

if RUN_TRAINING:
    for c in CANDIDATES:
        assert_supported_python(c)
        cmd = train_command(c)
        run(cmd, cwd=candidate_work_dir(c), execute=True)
else:
    print("\n[TRAINING] RUN_TRAINING=False; no training launched.")



[DATASET CHECK] C:\work\projects\myproj\SOLO_Supervised_RFDETR\Stat_Dataset\QA-2025v1_TwoClass_OVR_V2_20260618-101346


FileNotFoundError: Missing C:\work\projects\myproj\SOLO_Supervised_RFDETR\Stat_Dataset\QA-2025v1_TwoClass_OVR_V2_20260618-101346\train\_annotations.coco.json

## Before launching training

1. Confirm all three repositories clone cleanly.
2. Confirm requirements install in a Python 3.10/3.11 detector environment. Set `ALTDET_PYTHON=/path/to/env/bin/python`, or per-candidate overrides such as `ALTDET_DEIMV2_X_PYTHON=/path/to/env/bin/python`. Do not use the default UCloud Python 3.13 notebook kernel for installs/training.
3. Place private/licensed DEIMv2 DINOv3 backbone files in `DEIMv2/ckpts/` as required by the DEIMv2 README.
4. Confirm each generated config really points at `solo_twoclass_detection.yml`.
5. Inspect each generated config for dense-object caps: `num_queries`, `num_select`, `topk`, `max_detections`, or similar. For this dataset, aim for at least 600 candidate queries/detections where the repo supports it.
6. Keep the clinical test split out of the training config. Select with validation metrics only, then evaluate the single selected checkpoint on the forced clinical test set.